# Full Agentic Pipeline Demo

Decision -> Human Review -> Feedback Storage -> RL/Supervised Learning demo for Colab.

In [1]:
!pip install fastapi pydantic uvicorn pandas pyarrow scikit-learn torch transformers accelerate -q

In [2]:
import importlib
import os
import sys

repo_path = '/content/AI_Agentic_DL'
if not os.path.exists(repo_path):
    !git clone https://github.com/Lawapaul/AI_Agentic_DL.git {repo_path}

os.chdir(repo_path)
!git -C {repo_path} fetch origin
!git -C {repo_path} checkout codex/retraining-loop-system
!git -C {repo_path} pull origin codex/retraining-loop-system

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

importlib.invalidate_caches()
print('Repo ready:', repo_path)
print('feedback_store exists:', os.path.exists(os.path.join(repo_path, 'feedback_store')))
print('integration exists:', os.path.exists(os.path.join(repo_path, 'integration')))

Cloning into '/content/AI_Agentic_DL'...
remote: Enumerating objects: 598, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 598 (delta 177), reused 285 (delta 103), pack-reused 235 (from 1)
Receiving objects: 100% (598/598), 343.78 KiB | 4.52 MiB/s, done.
Resolving deltas: 100% (279/279), done.
Branch 'codex/retraining-loop-system' set up to track remote branch 'codex/retraining-loop-system' from 'origin'.
Switched to a new branch 'codex/retraining-loop-system'
From https://github.com/Lawapaul/AI_Agentic_DL
 * branch            codex/retraining-loop-system -> FETCH_HEAD
Already up to date.
Repo ready: /content/AI_Agentic_DL
feedback_store exists: True
integration exists: True


In [3]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier

path = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/data/processed/'
X = np.load(path + 'X_test.npy')[:20]
y = np.load(path + 'y_test.npy')[:20]
X = X.reshape(X.shape[0], -1)

model = RandomForestClassifier(n_estimators=10, random_state=42)
model.fit(X, y)
preds = model.predict(X)
probs = model.predict_proba(X)

In [5]:
attack_map = {
    0: 'Normal Traffic',
    1: 'DoS Attack',
    2: 'Port Scan',
    3: 'Brute Force Attack',
    4: 'Web Attack',
    5: 'Botnet',
    6: 'Infiltration',
}

events = []
for idx in range(5):
    confidence = float(max(probs[idx]))
    attack_name = attack_map.get(int(preds[idx]), 'Unknown Attack')
    suggested_action = 'No Action' if attack_name == 'Normal Traffic' else 'Block' if confidence > 0.85 else 'Alert' if confidence > 0.65 else 'Monitor'
    events.append({
        'event_id': f'event-{idx}',
        'decision_output': {
            'event_id': f'event-{idx}',
            'risk_score': confidence,
            'uncertainty': float(1.0 - confidence),
            'explanation': f'{attack_name} detected from CICIDS-derived test sample.',
            'suggested_action': suggested_action,
        },
    })

events[0]

{'event_id': 'event-0',
 'decision_output': {'event_id': 'event-0',
  'risk_score': 0.9,
  'uncertainty': 0.09999999999999998,
  'explanation': 'Normal Traffic detected from CICIDS-derived test sample.',
  'suggested_action': 'No Action'}}

In [6]:
from feedback_store.feedback_db import fetch_feedback, init_db
from integration.pipeline_connector import AgenticPipelineConnector

drive_results_dir = '/content/drive/MyDrive/Deep Learning Project/AI Agentic/experiments/results'
results_dir = drive_results_dir if os.path.exists('/content/drive/MyDrive') else os.path.join(repo_path, 'experiments', 'results')
os.makedirs(results_dir, exist_ok=True)
db_path = os.path.join(results_dir, 'feedback.db')
init_db(db_path)

connector = AgenticPipelineConnector(
    db_path=db_path,
    results_dir=results_dir,
    risk_threshold=0.7,
    uncertainty_threshold=0.25,
)

print('Feedback/results path:', results_dir)

In [7]:
run_outputs = []
for event in events:
    result = connector.run_full_pipeline(event)
    run_outputs.append(result)
    print('\nEvent:', result['decision_output']['event_id'])
    print('Decision:', result['decision_output'])
    print('Human Review:', result['review_output'])
    print('Feedback:', result['feedback_record'])


Event: event-0
Decision: {'event_id': 'event-0', 'risk_score': 0.9, 'uncertainty': 0.09999999999999998, 'explanation': 'Normal Traffic detected from CICIDS-derived test sample.', 'suggested_action': 'No Action'}
Human Review: {'approved': False, 'correct_action': 'Block', 'feedback': 'Adjusted action to Block based on risk and uncertainty.', 'severity_adjustment': 0.16, 'reviewer_id': 'simulated-analyst', 'reviewed_at': datetime.datetime(2026, 3, 29, 7, 47, 15, 193065)}
Feedback: {'id': 1, 'event_id': 'event-0', 'state': '{"risk_score": 0.9, "uncertainty": 0.09999999999999998, "suggested_action": "No Action", "explanation": "Normal Traffic detected from CICIDS-derived test sample."}', 'action_taken': 'No Action', 'human_action': 'Block', 'reward': -1.16, 'timestamp': '2026-03-29T07:47:15.193373+00:00'}

Event: event-1
Decision: {'event_id': 'event-1', 'risk_score': 0.8, 'uncertainty': 0.19999999999999996, 'explanation': 'Normal Traffic detected from CICIDS-derived test sample.', 'sugg

In [8]:
feedback_rows = fetch_feedback(db_path)
print('Stored feedback rows:', len(feedback_rows))
feedback_rows[:2]

Stored feedback rows: 5


[{'id': 5,
  'event_id': 'event-4',
  'state': '{"risk_score": 0.8, "uncertainty": 0.19999999999999996, "suggested_action": "No Action", "explanation": "Normal Traffic detected from CICIDS-derived test sample."}',
  'action_taken': 'No Action',
  'human_action': 'Alert',
  'reward': -1.12,
  'timestamp': '2026-03-29T07:47:15.736712+00:00'},
 {'id': 4,
  'event_id': 'event-3',
  'state': '{"risk_score": 1.0, "uncertainty": 0.0, "suggested_action": "No Action", "explanation": "Normal Traffic detected from CICIDS-derived test sample."}',
  'action_taken': 'No Action',
  'human_action': 'Block',
  'reward': -1.2,
  'timestamp': '2026-03-29T07:47:15.702326+00:00'}]

In [9]:
print('Updated RL policy:')
connector.rl_trainer.q_table

Updated RL policy:


{'risk_9_ctx_0': {'No Action': 0.0,
  'Monitor': 0.0,
  'Alert': 0.0,
  'Block': -0.4256},
 'risk_8_ctx_0': {'No Action': 0.0,
  'Monitor': 0.0,
  'Alert': -0.40320000000000006,
  'Block': 0.0},
 'risk_6_ctx_0': {'No Action': 0.0,
  'Monitor': -0.20800000000000002,
  'Alert': 0.0,
  'Block': 0.0}}